In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOK 62'S REAL POLICY (WHICH
#            ALREADY RECORDS EVERY REUSED PROBLEM 9/10 PATH)
# =============================================================================
import os
import sys
import json
import time
import gc
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebook 62's Real Policy")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB50_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_50_summary.json"
NB62_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_62_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB50_SUMMARY_PATH, "run 50_collections_optimization_business_understanding.ipynb first (Problem 9) -- "
                         "needed for Problem 8's real STATE_NAMES, reused verbatim"),
    (NB62_SUMMARY_PATH, "run 62_customer_intelligence_business_understanding.ipynb first (Problem 12)"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB50_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB50_SUMMARY = json.load(f)
with open(NB62_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB62_SUMMARY = json.load(f)

POLICY_PATH = Path(NB62_SUMMARY["policy_path"])
if not POLICY_PATH.exists():
    raise FileNotFoundError(f"{POLICY_PATH} not found.\nFix: re-run Notebook 62.")
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    CUSTOMER_INTELLIGENCE_POLICY = json.load(f)

UNIFIED_SCORE_WEIGHTS = CUSTOMER_INTELLIGENCE_POLICY["unified_score_weights"]
UNIFIED_RISK_GRADE_NAMES = CUSTOMER_INTELLIGENCE_POLICY["unified_risk_grade_names"]
UNIFIED_RISK_GRADE_CUT_PERCENTILES = CUSTOMER_INTELLIGENCE_POLICY["unified_risk_grade_cut_percentiles"]
KPI_TARGETS = CUSTOMER_INTELLIGENCE_POLICY["kpi_targets"]

# --- Problem 9's real persisted propensity-to-cure model + its own real
#     validated deployment policy (feature weights, monitored columns, cut
#     values, collections-eligible states, treatment-tier rule) -- every
#     value below is read from Notebook 62's own recorded paths, never
#     re-derived or guessed. ---
P9_REUSE = CUSTOMER_INTELLIGENCE_POLICY["reused_from_problem_9"]
P9_MODEL_PATH = Path(P9_REUSE["model_path"])
P9_DEPLOYMENT_POLICY_PATH = Path(P9_REUSE["deployment_policy_path"])
with open(P9_DEPLOYMENT_POLICY_PATH, "r", encoding="utf-8") as f:
    P9_DEPLOYMENT_POLICY = json.load(f)
MONITORED_COLS = sorted(P9_DEPLOYMENT_POLICY["monitored_features"])
P9_WEIGHTS = P9_DEPLOYMENT_POLICY["feature_weights"]["weights"]
P9_DIRECTIONS = P9_DEPLOYMENT_POLICY["feature_weights"]["directions"]
P9_MEANS = P9_DEPLOYMENT_POLICY["feature_weights"]["means"]
P9_STDS = P9_DEPLOYMENT_POLICY["feature_weights"]["stds"]
P9_CUT_LOW = P9_DEPLOYMENT_POLICY["cut_low"]
P9_CUT_HIGH = P9_DEPLOYMENT_POLICY["cut_high"]
COLLECTIONS_ELIGIBLE_STATES = P9_DEPLOYMENT_POLICY["collections_eligible_states"]
TREATMENT_TIER_RULE_NAMES = [t["name"] for t in P9_DEPLOYMENT_POLICY["treatment_tier_policy"]["tiers"]]

with open(Path(NB50_SUMMARY["policy_path"]), "r", encoding="utf-8") as f:
    _p9_business_policy = json.load(f)
STATE_NAMES = _p9_business_policy["reused_from_problem_8"]["state_names"]

# --- Problem 10's real, already-scored worklist -- the base population this
#     notebook joins everything else onto. Not re-derived: STATIC_PD,
#     DYNAMIC_PD, PD_TREND, RISK_LEVEL, TREND, ACTION, and the real target
#     label all come from here verbatim. ---
P10_REUSE = CUSTOMER_INTELLIGENCE_POLICY["reused_from_problem_10"]
P10_WORKLIST_PATH = Path(P10_REUSE["worklist_path"])
if not P10_WORKLIST_PATH.exists():
    raise FileNotFoundError(f"{P10_WORKLIST_PATH} not found.\nFix: re-run Notebook 55 (Problem 10).")

for _p, _label in [(P9_MODEL_PATH, "Problem 9's persisted propensity model")]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found ({_label}).\nFix: re-run Notebook 52 (Problem 9).")

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
DETECTED_TOTAL_RAM_BYTES = PROJECT_CONFIG["resource_limits"]["total_ram_bytes_detected"]

if "customer_intelligence_modeling" in PILLAR_DIRS:
    CUSTOMER_INTELLIGENCE_MODELING_DIR = PILLAR_DIRS["customer_intelligence_modeling"]
else:
    CUSTOMER_INTELLIGENCE_MODELING_DIR = (
        PROJECT_ROOT / "Phase5_Customer_Business_Intelligence"
        / "Problem12_360_Customer_Intelligence" / "modeling"
    )
    print(f"NOTE: 'customer_intelligence_modeling' not in pillar_dirs -- using fallback: "
          f"{CUSTOMER_INTELLIGENCE_MODELING_DIR}")
CUSTOMER_INTELLIGENCE_MODELING_DIR.mkdir(parents=True, exist_ok=True)
CUSTOMER_INTELLIGENCE_CHARTS_DIR = CUSTOMER_INTELLIGENCE_MODELING_DIR / "charts"
CUSTOMER_INTELLIGENCE_CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded Notebook 62's real policy from: {POLICY_PATH}")
print(f"Reused Problem 9's real model         : {P9_MODEL_PATH}")
print(f"Reused Problem 9's monitored columns   : {len(MONITORED_COLS)} ({MONITORED_COLS})")
print(f"Reused Problem 9's collections-eligible states: {COLLECTIONS_ELIGIBLE_STATES}")
print(f"Reused Problem 10's real worklist      : {P10_WORKLIST_PATH}")
print(f"UNIFIED_SCORE_WEIGHTS (from Notebook 62): {UNIFIED_SCORE_WEIGHTS}")
print(f"Modeling artifacts will be written under: {CUSTOMER_INTELLIGENCE_MODELING_DIR}")
print("\n✅ Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS (PHASE 4 CAP,
#            WITH THE TWO-TIER RAM GUARD ESTABLISHED AFTER THE REAL
#            NOTEBOOK 52 FREEZE)
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

_PHASE4_CPU_FRACTION_CAP = 0.92
_PHASE4_RAM_FRACTION_CAP = 0.92
_historical_thread_count = PROJECT_CONFIG["resource_limits"]["warp_thread_count"]
_historical_max_ram_bytes = PROJECT_CONFIG["resource_limits"]["max_ram_bytes"]
WARP_THREAD_COUNT = min(
    _historical_thread_count,
    max(1, round(DETECTED_LOGICAL_CORES * _PHASE4_CPU_FRACTION_CAP)),
)
MAX_RAM_BYTES = min(
    _historical_max_ram_bytes,
    round(DETECTED_TOTAL_RAM_BYTES * _PHASE4_RAM_FRACTION_CAP),
)

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from sklearn.metrics import (
        roc_auc_score, average_precision_score, accuracy_score, precision_score,
        recall_score, f1_score, log_loss, matthews_corrcoef, confusion_matrix, precision_recall_curve,
    )
except ImportError:
    missing.append("scikit-learn")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


def _available_ram_gb() -> float:
    return psutil.virtual_memory().available / 1e9


# --- Two-tier (warn/hard-fail) RAM pre-flight guard, established after the
#     real 45-minute freeze the user hit on Notebook 52 -- this notebook
#     does one real streaming pass over the raw CSV (Section 4), the same
#     class of operation that caused that freeze. ---
_available_ram_gb_at_start = _available_ram_gb()
_comfortable_available_ram_gb = 0.50 * (MAX_RAM_BYTES / 1e9)
_min_required_available_ram_gb = 0.25 * (MAX_RAM_BYTES / 1e9)
if _available_ram_gb_at_start < _min_required_available_ram_gb:
    raise RuntimeError(
        f"Only {_available_ram_gb_at_start:.2f} GB of system RAM is available, below the "
        f"{_min_required_available_ram_gb:.2f} GB floor this notebook needs. Close other Jupyter kernels / "
        f"memory-heavy applications, confirm with `psutil.virtual_memory().available / 1e9`, then re-run "
        f"this notebook from the top."
    )
if _available_ram_gb_at_start < _comfortable_available_ram_gb:
    print(f"⚠️  WARNING: only {_available_ram_gb_at_start:.2f} GB available "
          f"(comfortable margin is {_comfortable_available_ram_gb:.2f} GB). Proceeding.")
else:
    print(f"RAM pre-flight check passed: {_available_ram_gb_at_start:.2f} GB available >= "
          f"{_comfortable_available_ram_gb:.2f} GB comfortable margin.")

logger.info(f"Polars thread pool configured to {WARP_THREAD_COUNT}/{DETECTED_LOGICAL_CORES} threads")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
print(f"Configured RAM ceiling: {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n✅ Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    """Same 3(+1)-candidate resolver every notebook in this platform uses."""
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + "\nFix: run the notebook that produces this file again, or tell me the real path."
    )


TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)
print(f"Raw train_data.csv                                          : {RAW_TRAIN_DATA_PATH}")
print(f"train_split.csv (internal train, Notebook 02's real split)  : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv  (internal holdout, Notebook 02's real split): {TEST_SPLIT_PATH}")
print("\n✅ Section 3 complete.")


# =============================================================================
# SECTION 4: LOAD PROBLEM 10'S REAL SCORED WORKLIST (BASE POPULATION)
# =============================================================================
_section("SECTION 4: Load Problem 10's Real Scored Worklist (Base Population)")

BASE_DF = pl.read_parquet(P10_WORKLIST_PATH).rename({"TREND": "TREND_SEGMENT", "ACTION": "CREDIT_LINE_ACTION"})
print(f"Loaded Problem 10's real worklist: {BASE_DF.height:,} customers, columns: {BASE_DF.columns}")
print("\n✅ Section 4 complete.")


# =============================================================================
# SECTION 5: BUILD EACH CUSTOMER'S REAL LATEST-STATEMENT SEVERITY SCORE,
#            STATE, AND COLLECTIONS ELIGIBILITY (ONE REAL STREAMING PASS)
# =============================================================================
_section("SECTION 5: Build Each Customer's Real Latest-Statement Severity Score & State")

# --- Real, honest scope decision: Problem 9's propensity-to-cure model was
#     trained per-STATEMENT (predicting a state improvement at the very next
#     statement). For a per-CUSTOMER unified profile, this notebook scores
#     each collections-eligible customer's own real LATEST statement -- the
#     same "current status" framing this platform already used for
#     DYNAMIC_PD and RISK_LEVEL, and the one a live operational system would
#     actually use ("is this customer, right now, likely to cure"). ---
_schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
for _c in MONITORED_COLS:
    _schema_overrides[_c] = pl.Float32

_inf_clean_exprs = [
    pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
    for c in MONITORED_COLS
]

print(f"Streaming the real raw CSV once for each customer's real latest statement "
      f"({RAW_TRAIN_DATA_PATH.stat().st_size / 1e9:.2f} GB). RSS: {_rss_gb():.2f} GB, "
      f"available RAM: {_available_ram_gb():.2f} GB")
_t0 = time.time()
_latest_lf = (
    pl.scan_csv(RAW_TRAIN_DATA_PATH, schema_overrides=_schema_overrides)
    .with_row_index("_csv_row_order")
    .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
    .with_columns(_inf_clean_exprs)
    .sort(["customer_ID", "S_2", "_csv_row_order"])
    .group_by("customer_ID", maintain_order=False)
    .agg([pl.col(c).last().alias(c) for c in MONITORED_COLS])
)

_wz_cols = []
for _c in MONITORED_COLS:
    _mean, _std, _w, _d = P9_MEANS[_c], P9_STDS[_c], P9_WEIGHTS[_c], P9_DIRECTIONS[_c]
    if _std > 0 and _w > 0:
        _expr = ((pl.col(_c) - _mean) / _std * _w * _d).fill_null(0.0).alias(f"_wz_{_c}")
    else:
        _expr = pl.lit(0.0).alias(f"_wz_{_c}")
    _wz_cols.append(_expr)

_state_expr = (
    pl.when(pl.col("SEVERITY_SCORE") <= P9_CUT_LOW).then(pl.lit(STATE_NAMES[0]))
    .when(pl.col("SEVERITY_SCORE") <= P9_CUT_HIGH).then(pl.lit(STATE_NAMES[1]))
    .otherwise(pl.lit(STATE_NAMES[2]))
    .alias("STATE")
)

LATEST_STATEMENT_DF = (
    _latest_lf
    .with_columns(_wz_cols)
    .with_columns(pl.sum_horizontal([f"_wz_{c}" for c in MONITORED_COLS]).alias("SEVERITY_SCORE"))
    .with_columns(_state_expr)
    .select(["customer_ID", "SEVERITY_SCORE", "STATE"] + MONITORED_COLS)
    .collect(engine="streaming")
)
print(f"Built in {time.time() - _t0:.1f}s: {LATEST_STATEMENT_DF.height:,} customers' real latest-statement "
      f"severity score + state. RSS: {_rss_gb():.2f} GB, available RAM: {_available_ram_gb():.2f} GB")

LATEST_STATEMENT_DF = LATEST_STATEMENT_DF.with_columns(
    pl.col("STATE").is_in(COLLECTIONS_ELIGIBLE_STATES).alias("COLLECTIONS_ELIGIBLE")
)
_n_eligible = int(LATEST_STATEMENT_DF["COLLECTIONS_ELIGIBLE"].sum())
print(f"Real collections-eligible customers (latest statement in {COLLECTIONS_ELIGIBLE_STATES}): "
      f"{_n_eligible:,} / {LATEST_STATEMENT_DF.height:,} "
      f"({100.0 * _n_eligible / LATEST_STATEMENT_DF.height:.1f}%)")
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: SCORE COLLECTIONS-ELIGIBLE CUSTOMERS WITH PROBLEM 9'S REAL
#            PERSISTED MODEL (PROPENSITY-TO-CURE) -- HONEST NULLS ELSEWHERE
# =============================================================================
_section("SECTION 6: Score Collections-Eligible Customers With Problem 9's Real Persisted Model")

P9_MODEL = joblib.load(P9_MODEL_PATH)
_eligible_df = LATEST_STATEMENT_DF.filter(pl.col("COLLECTIONS_ELIGIBLE"))
_X_propensity = _eligible_df.select(MONITORED_COLS).to_numpy().astype(np.float32, copy=False)
_t0 = time.time()
_propensity_scores = P9_MODEL.predict_proba(_X_propensity)[:, 1] if len(_X_propensity) else np.array([])
print(f"Scored {len(_propensity_scores):,} real collections-eligible customers with Problem 9's real "
      f"persisted model in {time.time() - _t0:.1f}s.")

PROPENSITY_DF = _eligible_df.select(["customer_ID", "SEVERITY_SCORE"]).with_columns(
    pl.Series("PROPENSITY_TO_CURE", _propensity_scores, dtype=pl.Float64)
)

# --- Real treatment-tier policy (Notebook 50, reused verbatim): population-
#     median split on propensity-to-cure and on severity score, computed on
#     THIS notebook's own real collections-eligible population (a rule, not
#     a fixed cut value -- matches Notebook 50's own definition and
#     Notebook 51's own implementation exactly). ---
if PROPENSITY_DF.height:
    _median_propensity = float(PROPENSITY_DF["PROPENSITY_TO_CURE"].median())
    _median_severity = float(PROPENSITY_DF["SEVERITY_SCORE"].median())
    _tier_expr = (
        pl.when((pl.col("PROPENSITY_TO_CURE") < _median_propensity) & (pl.col("SEVERITY_SCORE") >= _median_severity))
        .then(pl.lit(TREATMENT_TIER_RULE_NAMES[0]))
        .when(pl.col("PROPENSITY_TO_CURE") >= _median_propensity)
        .then(pl.lit(TREATMENT_TIER_RULE_NAMES[1]))
        .otherwise(pl.lit(TREATMENT_TIER_RULE_NAMES[2]))
        .alias("TREATMENT_TIER")
    )
    PROPENSITY_DF = PROPENSITY_DF.with_columns(_tier_expr).drop("SEVERITY_SCORE")
else:
    _median_propensity, _median_severity = float("nan"), float("nan")
    PROPENSITY_DF = PROPENSITY_DF.with_columns(pl.lit(None, dtype=pl.Utf8).alias("TREATMENT_TIER")).drop(
        "SEVERITY_SCORE")

print(f"Real median propensity-to-cure (this run's eligible population): {_median_propensity:.4f}")
print(f"Real median severity score (this run's eligible population)    : {_median_severity:.4f}")
_tier_counts = PROPENSITY_DF.group_by("TREATMENT_TIER").agg(pl.len().alias("n")).sort("n", descending=True)
print("Real treatment-tier population counts:")
for _row in _tier_counts.iter_rows(named=True):
    print(f"  {str(_row['TREATMENT_TIER']):<20}: {_row['n']:>8,}")
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: JOIN EVERYTHING INTO THE REAL UNIFIED PROFILE -- HONEST NULLS
#            FOR THE COLLECTIONS-ONLY FIELDS
# =============================================================================
_section("SECTION 7: Join Everything Into the Real Unified Profile")

UNIFIED_DF = (
    BASE_DF
    .join(LATEST_STATEMENT_DF.select(["customer_ID", "COLLECTIONS_ELIGIBLE"]), on="customer_ID", how="left")
    .join(PROPENSITY_DF, on="customer_ID", how="left")
    .with_columns(pl.col("COLLECTIONS_ELIGIBLE").fill_null(False))
)
print(f"Real unified population (Problem 10's eligible base, left-joined with Problem 9's real signal): "
      f"{UNIFIED_DF.height:,} customers")
print(f"Of these, {int(UNIFIED_DF['COLLECTIONS_ELIGIBLE'].sum()):,} are real collections-eligible "
      f"(carry a real, non-null PROPENSITY_TO_CURE / TREATMENT_TIER); the remainder honestly carry a "
      f"null for both -- never imputed.")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: COMPUTE THE REAL UNIFIED_RISK_SCORE (NOTEBOOK 62'S COMPOSITE
#            FORMULA)
# =============================================================================
_section("SECTION 8: Compute the Real UNIFIED_RISK_SCORE")

_sw = UNIFIED_SCORE_WEIGHTS["static_pd_weight"]
_dw = UNIFIED_SCORE_WEIGHTS["dynamic_pd_weight"]
_cw = UNIFIED_SCORE_WEIGHTS["collections_adjustment_weight"]

# Where a real propensity score exists: 0.35*STATIC + 0.65*DYNAMIC + 0.10*(1-PROPENSITY), renormalized
# to sum to 1.0 across the three active terms. Where it does not: 0.35*STATIC + 0.65*DYNAMIC unchanged
# (weights already sum to 1.0 for those two terms alone -- see Notebook 62's verification check).
_with_propensity_score = (
    (_sw * pl.col("STATIC_PD") + _dw * pl.col("DYNAMIC_PD") + _cw * (1.0 - pl.col("PROPENSITY_TO_CURE")))
    / (_sw + _dw + _cw)
)
_without_propensity_score = _sw * pl.col("STATIC_PD") + _dw * pl.col("DYNAMIC_PD")

UNIFIED_DF = UNIFIED_DF.with_columns(
    pl.when(pl.col("PROPENSITY_TO_CURE").is_not_null())
    .then(_with_propensity_score)
    .otherwise(_without_propensity_score)
    .alias("UNIFIED_RISK_SCORE")
)
print(f"UNIFIED_RISK_SCORE real range: [{UNIFIED_DF['UNIFIED_RISK_SCORE'].min():.6f}, "
      f"{UNIFIED_DF['UNIFIED_RISK_SCORE'].max():.6f}], mean {UNIFIED_DF['UNIFIED_RISK_SCORE'].mean():.6f}")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: FIT REAL TERTILE CUTS (TRAIN SPLIT) & ASSIGN UNIFIED_RISK_GRADE
# =============================================================================
_section("SECTION 9: Fit Real Tertile Cuts (TRAIN Split) & Assign UNIFIED_RISK_GRADE")

TRAIN_IDS_DF = pl.read_csv(TRAIN_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8}).select("customer_ID")
TEST_IDS_DF = pl.read_csv(TEST_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8}).select("customer_ID")
TRAIN_DF = UNIFIED_DF.join(TRAIN_IDS_DF, on="customer_ID", how="inner")
HOLDOUT_DF = UNIFIED_DF.join(TEST_IDS_DF, on="customer_ID", how="inner")
print(f"Real internal TRAIN split (fits tertile cuts): {TRAIN_DF.height:,} customers")
print(f"Real internal HOLDOUT split (validates KPIs)  : {HOLDOUT_DF.height:,} customers")

_p_lo, _p_hi = UNIFIED_RISK_GRADE_CUT_PERCENTILES[0] / 100.0, UNIFIED_RISK_GRADE_CUT_PERCENTILES[1] / 100.0
UNIFIED_GRADE_CUT_LOW = float(TRAIN_DF["UNIFIED_RISK_SCORE"].quantile(_p_lo))
UNIFIED_GRADE_CUT_HIGH = float(TRAIN_DF["UNIFIED_RISK_SCORE"].quantile(_p_hi))
print(f"UNIFIED_RISK_GRADE cuts (fit on TRAIN, {UNIFIED_RISK_GRADE_CUT_PERCENTILES} percentiles): "
      f"low={UNIFIED_GRADE_CUT_LOW:.6f}, high={UNIFIED_GRADE_CUT_HIGH:.6f}")


def _assign_grade(df: "pl.DataFrame") -> "pl.DataFrame":
    _grade_expr = (
        pl.when(pl.col("UNIFIED_RISK_SCORE") <= UNIFIED_GRADE_CUT_LOW).then(pl.lit(UNIFIED_RISK_GRADE_NAMES[0]))
        .when(pl.col("UNIFIED_RISK_SCORE") <= UNIFIED_GRADE_CUT_HIGH).then(pl.lit(UNIFIED_RISK_GRADE_NAMES[1]))
        .otherwise(pl.lit(UNIFIED_RISK_GRADE_NAMES[2]))
        .alias("UNIFIED_RISK_GRADE")
    )
    return df.with_columns(_grade_expr)


UNIFIED_DF = _assign_grade(UNIFIED_DF)
TRAIN_DF = _assign_grade(TRAIN_DF)
HOLDOUT_DF = _assign_grade(HOLDOUT_DF)

_grade_counts = UNIFIED_DF.group_by("UNIFIED_RISK_GRADE").agg(pl.len().alias("n")).sort("UNIFIED_RISK_GRADE")
print("Real UNIFIED_RISK_GRADE population counts:")
for _row in _grade_counts.iter_rows(named=True):
    print(f"  {_row['UNIFIED_RISK_GRADE']:<12}: {_row['n']:>8,} ({100.0 * _row['n'] / UNIFIED_DF.height:.1f}%)")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: VALIDATE HARD-GATING KPIS ON THE REAL HOLDOUT SPLIT
# =============================================================================
_section("SECTION 10: Validate Hard-Gating KPIs on the Real HOLDOUT Split")

# --- profile_completeness: every customer must have a real, non-null value
#     for the seven always-applicable fields. Collections fields are
#     honestly allowed to be null. ---
_ALWAYS_APPLICABLE = ["STATIC_PD", "DYNAMIC_PD", "PD_TREND", "RISK_LEVEL", "TREND_SEGMENT",
                       "CREDIT_LINE_ACTION", "UNIFIED_RISK_SCORE"]
_null_counts = {c: int(UNIFIED_DF[c].is_null().sum()) for c in _ALWAYS_APPLICABLE}
PROFILE_COMPLETENESS_PASSED = all(v == 0 for v in _null_counts.values())
print(f"profile_completeness: null counts per always-applicable field: {_null_counts} -- "
      f"{'PASS' if PROFILE_COMPLETENESS_PASSED else 'FAIL'}")
print(f"  (Collections-only fields: {int(UNIFIED_DF['PROPENSITY_TO_CURE'].is_null().sum()):,} honest nulls "
      f"out of {UNIFIED_DF.height:,} -- this is a real data limitation, not a completeness failure.)")

# --- composite_non_inferiority: UNIFIED_RISK_SCORE's real holdout ROC-AUC
#     must be >= max(STATIC_PD AUC, DYNAMIC_PD AUC) - tolerance. ---
_y_holdout = HOLDOUT_DF["target"].to_numpy()
UNIFIED_ROC_AUC = float(roc_auc_score(_y_holdout, HOLDOUT_DF["UNIFIED_RISK_SCORE"].to_numpy()))
STATIC_PD_HOLDOUT_ROC_AUC = float(roc_auc_score(_y_holdout, HOLDOUT_DF["STATIC_PD"].to_numpy()))
DYNAMIC_PD_HOLDOUT_ROC_AUC = float(roc_auc_score(_y_holdout, HOLDOUT_DF["DYNAMIC_PD"].to_numpy()))
_best_single_signal_auc = max(STATIC_PD_HOLDOUT_ROC_AUC, DYNAMIC_PD_HOLDOUT_ROC_AUC)
_tolerance = KPI_TARGETS["composite_non_inferiority"]["auc_tolerance"]
COMPOSITE_NON_INFERIORITY_PASSED = UNIFIED_ROC_AUC >= (_best_single_signal_auc - _tolerance)

print(f"\nSTATIC_PD holdout ROC-AUC (real)   : {STATIC_PD_HOLDOUT_ROC_AUC:.4f}")
print(f"DYNAMIC_PD holdout ROC-AUC (real)  : {DYNAMIC_PD_HOLDOUT_ROC_AUC:.4f}")
print(f"UNIFIED_RISK_SCORE holdout ROC-AUC (real): {UNIFIED_ROC_AUC:.4f}")
print(f"composite_non_inferiority: {UNIFIED_ROC_AUC:.4f} >= {_best_single_signal_auc:.4f} - {_tolerance} = "
      f"{_best_single_signal_auc - _tolerance:.4f} -- {'PASS' if COMPOSITE_NON_INFERIORITY_PASSED else 'FAIL'}")

_expected_share_pct = 100.0 / len(UNIFIED_RISK_GRADE_NAMES)
_min_tier_pct = KPI_TARGETS["min_tier_population_pct"] / 100.0 * _expected_share_pct
_holdout_grade_counts = HOLDOUT_DF.group_by("UNIFIED_RISK_GRADE").agg(pl.len().alias("n"))
_undersized = [(r["UNIFIED_RISK_GRADE"], r["n"]) for r in _holdout_grade_counts.iter_rows(named=True)
               if 100.0 * r["n"] / HOLDOUT_DF.height < _min_tier_pct]
MIN_TIER_POPULATION_PASSED = len(_undersized) == 0
print(f"\nmin_tier_population_pct check: {'PASS' if MIN_TIER_POPULATION_PASSED else 'FAIL'} "
      f"(min real share required: {_min_tier_pct:.2f}% of HOLDOUT)")

ALL_HARD_GATES_PASSED = bool(PROFILE_COMPLETENESS_PASSED and COMPOSITE_NON_INFERIORITY_PASSED)
RECOMMENDED_FOR_PRODUCTION = bool(ALL_HARD_GATES_PASSED and MIN_TIER_POPULATION_PASSED)
if not ALL_HARD_GATES_PASSED:
    print(
        "\nHONEST FINDING: one or more hard-gating KPIs did not pass on this real run. This notebook still "
        "proceeds to report the full population (Sections 11-13 below) for completeness -- the same honest "
        "'not yet viable' standard Notebooks 36/40/48/56 held their own problems to -- but every downstream "
        "artifact and report for Problem 12 marks this run NOT RECOMMENDED FOR PRODUCTION."
    )
print(f"\nRECOMMENDED_FOR_PRODUCTION (this run): {RECOMMENDED_FOR_PRODUCTION}")
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: FULL CLASSIFICATION METRICS SUITE -- UNIFIED_RISK_SCORE
#             (STANDING RULE)
# =============================================================================
_section("SECTION 11: Full Classification Metrics Suite -- UNIFIED_RISK_SCORE (Real Holdout)")

_p_holdout = HOLDOUT_DF["UNIFIED_RISK_SCORE"].to_numpy()
UNIFIED_PR_AUC = float(average_precision_score(_y_holdout, _p_holdout))
UNIFIED_LOG_LOSS = float(log_loss(_y_holdout, _p_holdout, labels=[0, 1]))

_precisions, _recalls, _thresholds = precision_recall_curve(_y_holdout, _p_holdout)
_f1s = np.where((_precisions + _recalls) > 0, 2 * _precisions * _recalls / (_precisions + _recalls + 1e-12), 0.0)
_best_idx = int(np.argmax(_f1s[:-1])) if len(_thresholds) > 0 else 0
_f1_threshold = float(_thresholds[_best_idx]) if len(_thresholds) > 0 else 0.5
_y_pred = (_p_holdout >= _f1_threshold).astype(int)
_tn, _fp, _fn, _tp = confusion_matrix(_y_holdout, _y_pred, labels=[0, 1]).ravel()
UNIFIED_METRICS = {
    "roc_auc": UNIFIED_ROC_AUC, "pr_auc": UNIFIED_PR_AUC, "log_loss": UNIFIED_LOG_LOSS,
    "f1_optimal_threshold": _f1_threshold,
    "accuracy": float(accuracy_score(_y_holdout, _y_pred)),
    "precision": float(precision_score(_y_holdout, _y_pred, zero_division=0)),
    "recall": float(recall_score(_y_holdout, _y_pred, zero_division=0)),
    "f1": float(f1_score(_y_holdout, _y_pred, zero_division=0)),
    "specificity": float(_tn / (_tn + _fp)) if (_tn + _fp) > 0 else 0.0,
    "matthews_corrcoef": float(matthews_corrcoef(_y_holdout, _y_pred)),
    "confusion_matrix": {"tn": int(_tn), "fp": int(_fp), "fn": int(_fn), "tp": int(_tp)},
}
for _k, _v in UNIFIED_METRICS.items():
    print(f"  {_k:>20}: {_v}")
print("\n✅ Section 11 complete.")


# =============================================================================
# SECTION 12: CHARTS
# =============================================================================
_section("SECTION 12: Charts")

from sklearn.metrics import roc_curve as _roc_curve_fn
_fpr, _tpr, _ = _roc_curve_fn(_y_holdout, _p_holdout)
_fpr_s, _tpr_s, _ = _roc_curve_fn(_y_holdout, HOLDOUT_DF["STATIC_PD"].to_numpy())
_fpr_d, _tpr_d, _ = _roc_curve_fn(_y_holdout, HOLDOUT_DF["DYNAMIC_PD"].to_numpy())

plt.figure(figsize=(7, 5))
plt.plot(_fpr, _tpr, label=f"UNIFIED_RISK_SCORE (AUC={UNIFIED_ROC_AUC:.3f})", linewidth=2)
plt.plot(_fpr_s, _tpr_s, label=f"STATIC_PD alone (AUC={STATIC_PD_HOLDOUT_ROC_AUC:.3f})", linestyle=":")
plt.plot(_fpr_d, _tpr_d, label=f"DYNAMIC_PD alone (AUC={DYNAMIC_PD_HOLDOUT_ROC_AUC:.3f})", linestyle=":")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Problem 12 -- Composite Non-Inferiority: Unified vs. Single Input Signals (Real Holdout)")
plt.legend()
plt.tight_layout()
_roc_chart_path = CUSTOMER_INTELLIGENCE_CHARTS_DIR / "composite_non_inferiority_roc_chart.png"
plt.savefig(_roc_chart_path, dpi=120)
plt.close()

plt.figure(figsize=(7, 5))
_grade_order = UNIFIED_RISK_GRADE_NAMES
_default_rates = [
    float(UNIFIED_DF.filter(pl.col("UNIFIED_RISK_GRADE") == g)["target"].mean()) for g in _grade_order
]
plt.bar(_grade_order, _default_rates, color=["#2a9d8f", "#e9c46a", "#e76f51"])
plt.ylabel("Real Observed Default Rate")
plt.title("Problem 12 -- Real Default Rate by UNIFIED_RISK_GRADE")
plt.tight_layout()
_grade_chart_path = CUSTOMER_INTELLIGENCE_CHARTS_DIR / "unified_risk_grade_default_rate_chart.png"
plt.savefig(_grade_chart_path, dpi=120)
plt.close()

print(f"Wrote: {_roc_chart_path}")
print(f"Wrote: {_grade_chart_path}")
print("\n✅ Section 12 complete.")


# =============================================================================
# SECTION 13: PERSIST THE REAL UNIFIED PROFILE & MODELING RESULTS
# =============================================================================
_section("SECTION 13: Persist the Real Unified Profile & Modeling Results")

_profile_cols = ["customer_ID", "STATIC_PD", "DYNAMIC_PD", "PD_TREND", "RISK_LEVEL", "TREND_SEGMENT",
                  "CREDIT_LINE_ACTION", "COLLECTIONS_ELIGIBLE", "PROPENSITY_TO_CURE", "TREATMENT_TIER",
                  "UNIFIED_RISK_SCORE", "UNIFIED_RISK_GRADE", "target"]
UNIFIED_PROFILE = UNIFIED_DF.select(_profile_cols).sort("UNIFIED_RISK_SCORE", descending=True)
profile_path = CUSTOMER_INTELLIGENCE_MODELING_DIR / "unified_customer_profile.parquet"
UNIFIED_PROFILE.write_parquet(profile_path)
print(f"Wrote: {profile_path} ({profile_path.stat().st_size / 1e6:.1f} MB, {UNIFIED_PROFILE.height:,} customers)")

MODELING_RESULTS = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 12 -- 360 Degree Customer Intelligence (Modeling)",
    "eligible_population": UNIFIED_DF.height,
    "collections_eligible_population": int(UNIFIED_DF["COLLECTIONS_ELIGIBLE"].sum()),
    "train_split_population": TRAIN_DF.height,
    "holdout_split_population": HOLDOUT_DF.height,
    "unified_grade_cut_low": UNIFIED_GRADE_CUT_LOW,
    "unified_grade_cut_high": UNIFIED_GRADE_CUT_HIGH,
    "treatment_tier_median_propensity": _median_propensity,
    "treatment_tier_median_severity": _median_severity,
    "kpi_results": {
        "profile_completeness": {"null_counts": _null_counts, "passed": PROFILE_COMPLETENESS_PASSED},
        "composite_non_inferiority": {
            "unified_roc_auc": UNIFIED_ROC_AUC, "static_pd_roc_auc": STATIC_PD_HOLDOUT_ROC_AUC,
            "dynamic_pd_roc_auc": DYNAMIC_PD_HOLDOUT_ROC_AUC, "best_single_signal_auc": _best_single_signal_auc,
            "tolerance": _tolerance, "passed": COMPOSITE_NON_INFERIORITY_PASSED,
        },
        "min_tier_population_pct": {"passed": MIN_TIER_POPULATION_PASSED,
                                     "undersized_grades": [{"grade": g, "n": n} for g, n in _undersized]},
    },
    "all_hard_gates_passed": ALL_HARD_GATES_PASSED,
    "recommended_for_production": RECOMMENDED_FOR_PRODUCTION,
    "unified_metrics": UNIFIED_METRICS,
    "profile_path": str(profile_path),
    "random_seed": RANDOM_SEED,
}
modeling_results_path = CUSTOMER_INTELLIGENCE_MODELING_DIR / "customer_intelligence_modeling_results.json"
with open(modeling_results_path, "w", encoding="utf-8") as f:
    json.dump(MODELING_RESULTS, f, indent=2)
print(f"Wrote: {modeling_results_path}")
print("\n✅ Section 13 complete.")


# =============================================================================
# SECTION 14: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 14: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Unified profile file was written", profile_path.exists())
_all_checks_passed &= _check("Modeling results file was written", modeling_results_path.exists())
_all_checks_passed &= _check("UNIFIED_RISK_SCORE is finite and non-null for every customer",
                              int(UNIFIED_DF["UNIFIED_RISK_SCORE"].is_null().sum()) == 0
                              and bool(UNIFIED_DF["UNIFIED_RISK_SCORE"].is_finite().all()))
_all_checks_passed &= _check("Every customer was assigned a real UNIFIED_RISK_GRADE",
                              UNIFIED_DF["UNIFIED_RISK_GRADE"].is_in(UNIFIED_RISK_GRADE_NAMES).all())
_all_checks_passed &= _check("Collections-ineligible customers carry a null PROPENSITY_TO_CURE (honest, not "
                              "imputed)",
                              int(UNIFIED_DF.filter(~pl.col("COLLECTIONS_ELIGIBLE"))
                                  ["PROPENSITY_TO_CURE"].is_not_null().sum()) == 0)
_all_checks_passed &= _check("Collections-eligible customers carry a real, non-null PROPENSITY_TO_CURE",
                              int(UNIFIED_DF.filter(pl.col("COLLECTIONS_ELIGIBLE"))
                                  ["PROPENSITY_TO_CURE"].is_null().sum()) == 0)
_all_checks_passed &= _check("Profile row count matches the eligible population count",
                              UNIFIED_PROFILE.height == UNIFIED_DF.height)
_all_checks_passed &= _check("UNIFIED_RISK_GRADE cut values are correctly ordered (low < high)",
                              UNIFIED_GRADE_CUT_LOW < UNIFIED_GRADE_CUT_HIGH)
_all_checks_passed &= _check("UNIFIED_RISK_SCORE ROC-AUC is a real value in (0.5, 1.0]",
                              0.5 < UNIFIED_ROC_AUC <= 1.0)
_all_checks_passed &= _check("Train and holdout splits do not overlap",
                              len(set(TRAIN_DF["customer_ID"]) & set(HOLDOUT_DF["customer_ID"])) == 0)
_all_checks_passed &= _check("KPI results object reused the exact same real AUC values computed above "
                              "(no re-derivation)",
                              MODELING_RESULTS["kpi_results"]["composite_non_inferiority"]["unified_roc_auc"]
                              == UNIFIED_ROC_AUC)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 14 complete -- all checks passed.")


# =============================================================================
# SECTION 15: WRITE NOTEBOOK 63 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 15: Write Notebook 63 Summary Artifact")

NB63_SUMMARY = {
    "notebook": "63_customer_intelligence_modeling.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "modeling_results_path": str(modeling_results_path),
    "profile_path": str(profile_path),
    "eligible_population": UNIFIED_DF.height,
    "collections_eligible_population": int(UNIFIED_DF["COLLECTIONS_ELIGIBLE"].sum()),
    "unified_grade_cut_low": UNIFIED_GRADE_CUT_LOW,
    "unified_grade_cut_high": UNIFIED_GRADE_CUT_HIGH,
    "all_hard_gates_passed": ALL_HARD_GATES_PASSED,
    "recommended_for_production": RECOMMENDED_FOR_PRODUCTION,
    "unified_roc_auc": UNIFIED_ROC_AUC,
    "best_single_signal_auc": _best_single_signal_auc,
    "warp_thread_count": WARP_THREAD_COUNT,
    "max_ram_bytes": MAX_RAM_BYTES,
    "random_seed": RANDOM_SEED,
}
NB63_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_63_summary.json"
with open(NB63_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB63_SUMMARY, f, indent=2)
print(f"Wrote: {NB63_SUMMARY_PATH}")

_section("NOTEBOOK 63 COMPLETE")
print(f"Unified population (real)                     : {UNIFIED_DF.height:,} customers")
print(f"Collections-eligible (real)                    : {int(UNIFIED_DF['COLLECTIONS_ELIGIBLE'].sum()):,}")
print(f"profile_completeness                           : {'PASS' if PROFILE_COMPLETENESS_PASSED else 'FAIL'}")
print(f"composite_non_inferiority                      : "
      f"{'PASS' if COMPOSITE_NON_INFERIORITY_PASSED else 'FAIL'} "
      f"(unified={UNIFIED_ROC_AUC:.4f} vs. best single={_best_single_signal_auc:.4f})")
print(f"RECOMMENDED_FOR_PRODUCTION (this run)          : {RECOMMENDED_FOR_PRODUCTION}")
print(f"Unified profile written to: {profile_path}")
print(
    "\nNext: 64_customer_intelligence_validation_deployment.ipynb -- independently reproduces this notebook's "
    "real pipeline from scratch, bootstraps a confidence interval on the composite_non_inferiority gap, "
    "verifies the persisted unified profile against a fresh reproduction, and packages the real FastAPI "
    "unified-profile lookup service."
)
